In [17]:
!pip install -q transformers datasets scikit-learn

In [18]:
import os
import re
import pandas as pd
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight

from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

# =========================
# 공통 설정
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAME = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LENGTH = 256

# NS: 1 = 문장형(S)
AXIS_THRESHOLDS = {
    "NS": 0.56,
}

print("device:", device)
print("MODEL_NAME:", MODEL_NAME)
print("MAX_LENGTH:", MAX_LENGTH)
print("AXIS_THRESHOLDS:", AXIS_THRESHOLDS)


device: cuda
MODEL_NAME: klue/bert-base
MAX_LENGTH: 256
AXIS_THRESHOLDS: {'NS': 0.56}


In [19]:
def train_axis_model(csv_path, save_dir):
    df = pd.read_csv(csv_path).dropna()
    df.columns = ['review_text', 'label']
    df['label'] = df['label'].astype(int)

    # 저장 폴더 미리 생성
    os.makedirs(save_dir, exist_ok=True)

    # ✅ 1. train/validation split을 여기서 한 번만 만든다
    train_df, val_df = train_test_split(
        df,
        test_size=0.2,
        random_state=42,
        stratify=df['label']   # A/E 비율이 train과 val에 비슷하게 들어가도록 함
    )

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)

    # ✅ 2. 나중에 오분류 분석할 때 같은 검증셋을 쓰기 위해 저장
    train_df.to_csv(f"{save_dir}/train_split.csv", index=False, encoding="utf-8-sig")
    val_df.to_csv(f"{save_dir}/val_split.csv", index=False, encoding="utf-8-sig")

    print(f"학습셋 크기: {len(train_df)}개")
    print(f"검증셋 크기: {len(val_df)}개")
    print("검증셋 라벨 분포")
    print(val_df['label'].value_counts().sort_index())

    # ✅ 3. class weight는 전체 df가 아니라 train_df 기준으로 계산하는 게 더 안전함
    class_weights = compute_class_weight(
        'balanced',
        classes=np.unique(train_df['label']),
        y=train_df['label']
    )
    weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

    # ✅ 4. sklearn으로 나눈 train_df, val_df를 HuggingFace Dataset으로 변환
    train_dataset = Dataset.from_pandas(train_df)
    val_dataset = Dataset.from_pandas(val_df)

    def tokenize_fn(x):
        return tokenizer(
            x["review_text"],
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH
        )

    tokenized_train = train_dataset.map(tokenize_fn, batched=True)
    tokenized_val = val_dataset.map(tokenize_fn, batched=True)

    def compute_metrics(pred):
        labels = pred.label_ids
        preds = pred.predictions.argmax(-1)

        return {
            "accuracy": accuracy_score(labels, preds),
            "f1_macro": f1_score(labels, preds, average="macro"),
            "f1_binary": f1_score(labels, preds, average="binary"),
            "balanced_accuracy": balanced_accuracy_score(labels, preds)
        }

    class CustomTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.pop("labels")
            outputs = model(**inputs)
            loss_fct = nn.CrossEntropyLoss(weight=weights_tensor)
            loss = loss_fct(outputs.logits.view(-1, 2), labels.view(-1))
            return (loss, outputs) if return_outputs else loss

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2
    ).to(device)

    args = TrainingArguments(
      output_dir=save_dir,
      eval_strategy="epoch",
      save_strategy="epoch",
      learning_rate=2e-5,
      num_train_epochs=4,
      per_device_train_batch_size=16,
      load_best_model_at_end=True,
      metric_for_best_model="f1_macro",
      greater_is_better=True
  )

    trainer = CustomTrainer(
        model=model,
        args=args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        compute_metrics=compute_metrics
    )

    trainer.train()

    trainer.save_model(save_dir)
    tokenizer.save_pretrained(save_dir)

    print(f"✅ 모델 저장 완료: {save_dir}")
    print(f"✅ 검증셋 저장 완료: {save_dir}/val_split.csv")


In [20]:
# =========================
# NS 모델만 학습
# =========================
NS_CSV_PATH = "/content/train_NS.csv"
NS_result_df_SAVE_DIR = "./model_axis2_NS"

train_axis_model(
    NS_CSV_PATH,
    NS_result_df_SAVE_DIR
)

학습셋 크기: 2400개
검증셋 크기: 600개
검증셋 라벨 분포
label
0    300
1    300
Name: count, dtype: int64


Map:   0%|          | 0/2400 [00:00<?, ? examples/s]

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on you

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Binary,Balanced Accuracy
1,No log,0.319425,0.855000,0.854706,0.848168,0.855000
2,No log,0.354741,0.841667,0.841645,0.843493,0.841667
3,No log,0.467148,0.868333,0.867884,0.860177,0.868333
4,0.250871,0.523090,0.858333,0.858087,0.852174,0.858333


... (3 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ 모델 저장 완료: ./model_axis2_NS
✅ 검증셋 저장 완료: ./model_axis2_NS/val_split.csv


In [21]:
# =========================
# NS 모델 로드
# =========================
model_ns = AutoModelForSequenceClassification.from_pretrained("./model_axis2_NS").to(device)
model_ns.eval()

NS_THRESHOLD = AXIS_THRESHOLDS["NS"]

print("✅ NS 모델 로드 완료")
print("NS_THRESHOLD:", NS_THRESHOLD)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ NS 모델 로드 완료
NS_THRESHOLD: 0.56


In [22]:
# =========================
# NS 문장 단위 단서 기반 보정 규칙
# 기존 방식 유지: 모델 예측 + 문장 단위 cue 분석 + N/S 보정
# =========================

N_CUES = [
    "이야기", "줄거리", "서사", "전개", "흐름", "사건",
    "장면들이", "인물", "주인공", "등장인물", "관계",
    "갈등", "결말", "과정", "변화", "성장",
    "전체 이야기", "전체 흐름", "이야기 전체", "책 전체",
    "처음부터 끝까지", "전반적으로", "구성", "구조",
    "이어진다", "펼쳐진다", "진행된다", "흘러간다",
    "서사가", "스토리", "플롯"
]

S_CUES = [
    "문장", "구절", "표현", "한 문장", "이 문장",
    "그 문장", "문구", "대사", "인용", "말",
    "단어", "한 구절", "짧은 문장", "마지막 문장",
    "첫 문장", "기억에 남는 문장", "인상 깊은 문장",
    "계속 떠오르는 문장", "계속 맴도는", "곱씹게",
    "밑줄", "필사", "적어두고 싶은", "오래 남는다",
    "머릿속에 남는다", "마음에 남는다"
]


def split_sentences(text):
    sentences = re.split(r'(?<=[.!?。！？])\s+', text.strip())
    sentences = [s.strip() for s in sentences if s.strip()]
    return sentences if sentences else [text]


def count_cues(text, cues):
    return sum(1 for cue in cues if cue in text)


def get_ns_base_prediction(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    ).to(device)

    with torch.no_grad():
        outputs = model_ns(**inputs)

    probs = F.softmax(outputs.logits, dim=-1)[0]
    prob_N = probs[0].item()
    prob_S = probs[1].item()
    base_pred = 1 if prob_S >= NS_THRESHOLD else 0

    return base_pred, prob_N, prob_S


def analyze_ns_sentences(text):
    sentences = split_sentences(text)
    rows = []

    for idx, sent in enumerate(sentences, start=1):
        n_count = count_cues(sent, N_CUES)
        s_count = count_cues(sent, S_CUES)

        if n_count > s_count:
            cue_label = "서사형(N)"
        elif s_count > n_count:
            cue_label = "문장형(S)"
        else:
            cue_label = "중립/불명확"

        rows.append({
            "문장번호": idx,
            "문장": sent,
            "N_cue_count": n_count,
            "S_cue_count": s_count,
            "문장단서판단": cue_label,
        })

    return pd.DataFrame(rows)


def predict_ns_with_context_rule(text):
    """
    N/S 전용 보정 예측.
    기본 모델은 그대로 쓰되,
    문장 단위 cue 분석을 통해 서사/문장 중심성을 보정한다.
    """

    # 1. 전체 리뷰 기준 모델 예측
    base_pred, prob_N, prob_S = get_ns_base_prediction(text)

    # 2. 문장 단위 cue 분석
    sentence_df = analyze_ns_sentences(text)

    n_sentence_count = (sentence_df["문장단서판단"] == "서사형(N)").sum()
    s_sentence_count = (sentence_df["문장단서판단"] == "문장형(S)").sum()

    total_n_cues = count_cues(text, N_CUES)
    total_s_cues = count_cues(text, S_CUES)

    # 3. 보정 규칙

    # 모델은 S라고 했지만 → N으로 보정
    if base_pred == 1:
        if (
            n_sentence_count >= 2
            and total_n_cues >= total_s_cues + 2
            and total_n_cues >= 4
        ):
            corrected_pred = 0
            correction_reason = "N/S 보정: 이야기 흐름·전개·인물 변화 중심 서술이 리뷰 중심"
        else:
            corrected_pred = base_pred
            correction_reason = "모델 예측 유지"

    # 모델은 N이라고 했지만 → S로 보정
    elif base_pred == 0:
        if (
            s_sentence_count >= 2
            and total_s_cues >= total_n_cues + 2
            and total_s_cues >= 4
        ):
            corrected_pred = 1
            correction_reason = "N/S 보정: 특정 문장·구절·표현 중심 서술이 리뷰 중심"
        else:
            corrected_pred = base_pred
            correction_reason = "모델 예측 유지"

    else:
        corrected_pred = base_pred
        correction_reason = "모델 예측 유지"

    conf = prob_S if corrected_pred == 1 else prob_N

    return {
        "pred": corrected_pred,
        "label": "서사형(N)" if corrected_pred == 0 else "문장형(S)",
        "confidence": conf,
        "prob_N": prob_N,
        "prob_S": prob_S,
        "base_pred": base_pred,
        "base_label": "서사형(N)" if base_pred == 0 else "문장형(S)",
        "correction_reason": correction_reason,
        "n_sentence_count": int(n_sentence_count),
        "s_sentence_count": int(s_sentence_count),
        "total_n_cues": int(total_n_cues),
        "total_s_cues": int(total_s_cues),
        "sentence_df": sentence_df,
    }

In [23]:
# =========================
# 보기 좋은 단일 리뷰 출력 함수
# =========================

def display_ns_prediction(text):
    result = predict_ns_with_context_rule(text)

    summary_df = pd.DataFrame([{
        "최종 예측": result["label"],
        "최종 확신도": f"{result['confidence'] * 100:.1f}%",
        "서사형(N) 확률": f"{result['prob_N'] * 100:.1f}%",
        "문장형(S) 확률": f"{result['prob_S'] * 100:.1f}%",
        "모델 기본 예측": result["base_label"],
        "보정 사유": result["correction_reason"],
        "서사형 문장 수": result["n_sentence_count"],
        "문장형 문장 수": result["s_sentence_count"],
        "N 단서 수": result["total_n_cues"],
        "S 단서 수": result["total_s_cues"],
    }])

    print("=" * 80)
    print("입력 리뷰")
    print("=" * 80)
    print(text.strip())

    print("\n" + "=" * 80)
    print("NS 최종 예측 요약")
    print("=" * 80)
    display(summary_df)

    print("\n" + "=" * 80)
    print("감상 단위 분석")
    print("=" * 80)
    display(result["sentence_df"])

    return result


In [33]:
# =========================
# 테스트 리뷰
# =========================

test_review = """
이 책은 인물의 관계 변화를 중심으로 이야기를 전개하며, 초반의 갈등이 후반부 결말까지 자연스럽게 이어진다. 특히 각 장면이 작품의 주제를 설명하는 방식으로 배치되어 있어 전체 구성이 안정적으로 느껴진다. 읽는 동안 주인공의 선택이 내 상황과 겹쳐 보여 조금 먹먹했다.
"""


result = display_ns_prediction(test_review)


입력 리뷰
이 책은 인물의 관계 변화를 중심으로 이야기를 전개하며, 초반의 갈등이 후반부 결말까지 자연스럽게 이어진다. 특히 각 장면이 작품의 주제를 설명하는 방식으로 배치되어 있어 전체 구성이 안정적으로 느껴진다. 읽는 동안 주인공의 선택이 내 상황과 겹쳐 보여 조금 먹먹했다.

NS 최종 예측 요약


,최종 예측,최종 확신도,서사형(N) 확률,문장형(S) 확률,모델 기본 예측,보정 사유,서사형 문장 수,문장형 문장 수,N 단서 수,S 단서 수
0,서사형(N),91.8%,91.8%,8.2%,서사형(N),모델 예측 유지,3,0,10,1



감상 단위 분석


,문장번호,문장,N_cue_count,S_cue_count,문장단서판단
0,1,"이 책은 인물의 관계 변화를 중심으로 이야기를 전개하며, 초반의 갈등이 후반부 결말...",8,1,서사형(N)
1,2,특히 각 장면이 작품의 주제를 설명하는 방식으로 배치되어 있어 전체 구성이 안정적으...,1,0,서사형(N)
2,3,읽는 동안 주인공의 선택이 내 상황과 겹쳐 보여 조금 먹먹했다.,1,0,서사형(N)


In [25]:
# =========================
# NS 검증셋 성능 / 오분류 요약
# =========================

def predict_ns_batch(texts, batch_size=32, max_length=MAX_LENGTH):
    model_ns.eval()

    preds = []
    probs_N = []
    probs_S = []
    confidences = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=max_length
        ).to(device)

        with torch.no_grad():
            logits = model_ns(**inputs).logits

        prob = F.softmax(logits, dim=-1).cpu()
        prob_N = prob[:, 0]
        prob_S = prob[:, 1]

        pred = (prob_N >= NS_THRESHOLD).long()
        confidence = torch.where(pred == 1, prob_N, prob_S)

        preds.extend(pred.tolist())
        probs_N.extend(prob_N.tolist())
        probs_S.extend(prob_S.tolist())
        confidences.extend(confidence.tolist())

    return preds, probs_N, probs_S, confidences


def analyze_ns_validation(val_csv_path="./model_axis2_NS/val_split.csv", save_dir="/content"):
    print("\n" + "=" * 70)
    print("NS 축 성능 / 오분류 요약")
    print("=" * 70)

    val_df = pd.read_csv(val_csv_path)
    val_df["label"] = val_df["label"].astype(int)

    label_counts = val_df["label"].value_counts().sort_index()
    print("\n[1] 검증셋 라벨 분포")
    print(f"  서사형(N=0): {label_counts.get(0, 0)}개")
    print(f"  문장형(S=1): {label_counts.get(1, 0)}개")
    print(f"  전체: {len(val_df)}개")

    texts = val_df["review_text"].tolist()
    preds, probs_N, probs_S, confidences = predict_ns_batch(texts)

    val_df["pred"] = preds
    val_df["prob_N"] = probs_N
    val_df["prob_S"] = probs_S
    val_df["confidence"] = confidences
    val_df["correct"] = val_df["label"] == val_df["pred"]

    y_true = val_df["label"]
    y_pred = val_df["pred"]

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    print("\n[2] 전체 성능")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  Macro F1 : {macro_f1:.4f}")

    print("\n[3] Confusion Matrix")
    print("       예측 N    예측 S")
    print(f"실제 N   {cm[0,0]:>5}    {cm[0,1]:>5}")
    print(f"실제 S   {cm[1,0]:>5}    {cm[1,1]:>5}")

    print("\n[4] 오분류 방향")
    print(f"  서사형(N) → 문장형(N): {cm[0,1]}개")
    print(f"  문장형(S) → 서사형(S): {cm[1,0]}개")

    print("\n[5] 상세 리포트")
    print(classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["서사형(N)", "문장형(S)"],
        digits=4
    ))

    os.makedirs(save_dir, exist_ok=True)
    result_path = f"{save_dir}/prediction_result_NS.csv"
    val_df.to_csv(result_path, index=False, encoding="utf-8-sig")

    print("\n[6] 저장 완료")
    print(f"  전체 예측 결과: {result_path}")

    return val_df


ns_result_df = analyze_ns_validation()



NS 축 성능 / 오분류 요약

[1] 검증셋 라벨 분포
  서사형(N=0): 300개
  문장형(S=1): 300개
  전체: 600개

[2] 전체 성능
  Accuracy : 0.1383
  Macro F1 : 0.1360

[3] Confusion Matrix
       예측 N    예측 S
실제 N      26      274
실제 S     243       57

[4] 오분류 방향
  서사형(N) → 문장형(N): 274개
  문장형(S) → 서사형(S): 243개

[5] 상세 리포트
              precision    recall  f1-score   support

      서사형(N)     0.0967    0.0867    0.0914       300
      문장형(S)     0.1722    0.1900    0.1807       300

    accuracy                         0.1383       600
   macro avg     0.1344    0.1383    0.1360       600
weighted avg     0.1344    0.1383    0.1360       600


[6] 저장 완료
  전체 예측 결과: /content/prediction_result_NS.csv


In [26]:
# =========================
# 여러 리뷰를 한 번에 예측하고 표로 보기
# =========================

def predict_ns_reviews_to_table(reviews):
    rows = []
    for i, text in enumerate(reviews, start=1):
        result = predict_ns_with_context_rule(text)
        rows.append({
            "번호": i,
            "리뷰": text.strip().replace("\n", " ")[:120],
            "최종 예측": result["label"],
            "서사형(N) 확률": result["prob_N"],
            "문장형(S) 확률": result["prob_S"],
            "보정 사유": result["correction_reason"],
            "서사형 문장 수": result["n_sentence_count"],
            "문장형 문장 수": result["s_sentence_count"],
        })
    return pd.DataFrame(rows)

sample_reviews = [
    "이 책은 인간관계의 원리를 사례를 통해 체계적으로 설명한다. 각 장의 구성도 명확하고 핵심 개념도 잘 정리되어 있다.",
    "읽는 내내 내가 겪었던 일이 떠올라서 마음이 아팠다. 주인공의 감정이 너무 잘 느껴져서 깊이 공감했다.",
    "이 책은 인간관계의 원리를 사례를 통해 체계적으로 설명한다. 각 장의 구성이 명확하다. 다만 내 경험이 떠올라 조금 씁쓸했다.",
]

sample_result_df = predict_ns_reviews_to_table(sample_reviews)
display(sample_result_df)


,번호,리뷰,최종 예측,서사형(N) 확률,문장형(S) 확률,보정 사유,서사형 문장 수,문장형 문장 수
0,1,이 책은 인간관계의 원리를 사례를 통해 체계적으로 설명한다. 각 장의 구성도 명확하...,서사형(N),0.973168,0.026832,모델 예측 유지,2,0
1,2,읽는 내내 내가 겪었던 일이 떠올라서 마음이 아팠다. 주인공의 감정이 너무 잘 느껴...,서사형(N),0.992890,0.007110,모델 예측 유지,1,0
2,3,이 책은 인간관계의 원리를 사례를 통해 체계적으로 설명한다. 각 장의 구성이 명확하...,문장형(S),0.346041,0.653959,모델 예측 유지,2,0
